In [11]:
import os, glob, time, random
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import torchvision.models as models
import torchvision.transforms.functional as TF
from sklearn.metrics import roc_curve, auc as sk_auc
import matplotlib
matplotlib.use("Agg")          # headless — no display needed
import matplotlib.pyplot as plt

In [12]:
try:
    from tqdm import tqdm
    HAS_TQDM = True
except ImportError:
    HAS_TQDM = False

In [ ]:
# 1.  Configuration
CFG = dict(
    train_lenses    = "dataset/train_lenses",
    train_nonlenses = "dataset/train_nonlenses",
    test_lenses     = "dataset/test_lenses",
    test_nonlenses  = "dataset/test_nonlenses",

    # ── Key change: small, balanced epochs ──
    # Each epoch sees 2 * N_pos samples (N_pos positive + N_pos negative).
    # For 1730 lenses → 3460 samples/epoch.  30 epochs ≈ 100k samples total.
    # v1 saw 30 * 30405 ≈ 900k — nearly 9x more work for no gain.
    epochs           = 40,
    batch_size       = 32,          # smaller = more gradient steps per epoch

    # Learning rate schedule
    lr_head          = 3e-4,        # FC head (randomly initialised)
    lr_backbone      = 5e-5,        # ResNet backbone (pretrained)

    weight_decay     = 1e-4,
    num_workers      = 0,           # set to 4 if you have enough RAM
    seed             = 42,
    patience         = 7,           # early stopping

    checkpoint       = "best_model_v2.pth",
    roc_plot         = "roc_curve_v2.png",
    device           = "cuda" if torch.cuda.is_available() else "cpu",
    use_amp          = torch.cuda.is_available(),   # mixed precision on GPU only
)

In [14]:
# 2.  Seed
def set_seed(s):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)

set_seed(CFG["seed"])
print(f"Device : {CFG['device']}  |  AMP : {CFG['use_amp']}")

Device : cpu  |  AMP : False


In [15]:
# 3.  Dataset
class LensDataset(Dataset):
    """
    Loads (3,64,64) .npy files.
    Per-channel min-max → [0,1] then ImageNet normalisation.
    Augmentation: hflip + vflip + rot90 + small random rotation.
    NOTE: augmentation is geometric only — NO colour jitter, because
    g/r/i photometric bands encode physical information.
    """
    MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3,1,1)
    STD  = torch.tensor([0.229, 0.224, 0.225]).view(3,1,1)

    def __init__(self, lens_dir, nonlens_dir, augment=False):
        self.augment = augment
        lens_f    = sorted(glob.glob(os.path.join(lens_dir,    "*.npy")))
        nonlens_f = sorted(glob.glob(os.path.join(nonlens_dir, "*.npy")))
        if not lens_f:
            raise FileNotFoundError(f"No .npy files in {lens_dir}")
        if not nonlens_f:
            raise FileNotFoundError(f"No .npy files in {nonlens_dir}")
        self.files  = lens_f + nonlens_f
        self.labels = [1]*len(lens_f) + [0]*len(nonlens_f)
        n_pos, n_neg = len(lens_f), len(nonlens_f)
        print(f"  lenses={n_pos}  non-lenses={n_neg}  "
              f"ratio={n_neg/max(n_pos,1):.1f}:1  augment={augment}")

    def __len__(self):
        return len(self.files)

    def _normalise(self, arr):
        t = torch.from_numpy(arr.astype(np.float32))   # (3,H,W)
        for c in range(3):
            lo, hi = t[c].min(), t[c].max()
            if hi > lo:
                t[c] = (t[c] - lo) / (hi - lo)
        return (t - self.MEAN) / self.STD

    def __getitem__(self, idx):
        t = self._normalise(np.load(self.files[idx]))
        if self.augment:
            if random.random() > 0.5: t = TF.hflip(t)
            if random.random() > 0.5: t = TF.vflip(t)
            t = torch.rot90(t, random.randint(0,3), dims=[1,2])
            t = TF.rotate(t, random.uniform(-15, 15))
        return t, torch.tensor(self.labels[idx], dtype=torch.float32)

In [ ]:
# 4.  Balanced sampler
def make_balanced_sampler(dataset):
    """
    Samples exactly 2 * N_pos items per epoch:
      N_pos positives (all seen) + N_pos negatives (random subset).
    This makes each epoch fast AND perfectly balanced.
    """
    labels   = np.array(dataset.labels)
    n_pos    = int(labels.sum())
    n_neg    = len(labels) - n_pos
    n_epoch  = 2 * n_pos          # epoch size: small and balanced

    # Per-sample weight: uniform within each class
    w = np.where(labels == 1, 1.0/n_pos, 1.0/n_neg)
    print(f"  Epoch size: {n_epoch} samples  ({n_pos} pos + {n_pos} neg drawn from {n_neg} neg)")
    return WeightedRandomSampler(
        weights     = torch.from_numpy(w).float(),
        num_samples = n_epoch,
        replacement = True,
    )

In [17]:
# 5.  Model
def build_model(device):
    """
    ResNet18, pretrained.
    Uses differential learning rates:
      backbone → lr_backbone  (fine-tune carefully)
      FC head  → lr_head      (train from scratch, needs higher LR)
    """
    m = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
    m.fc = nn.Linear(m.fc.in_features, 1)
    return m.to(device)

def make_optimizer(model, cfg):
    backbone_params = [p for n,p in model.named_parameters() if "fc" not in n]
    head_params     = list(model.fc.parameters())
    return torch.optim.AdamW([
        {"params": backbone_params, "lr": cfg["lr_backbone"]},
        {"params": head_params,     "lr": cfg["lr_head"]},
    ], weight_decay=cfg["weight_decay"])

In [18]:
# 6.  Training
def train(cfg):
    device = cfg["device"]

    print("\n── Datasets ──")
    train_ds = LensDataset(cfg["train_lenses"], cfg["train_nonlenses"], augment=True)
    test_ds  = LensDataset(cfg["test_lenses"],  cfg["test_nonlenses"],  augment=False)

    # pos_weight for loss  (computed from TRAINING distribution)
    labels_arr = np.array(train_ds.labels)
    n_pos = int(labels_arr.sum())
    n_neg = len(labels_arr) - n_pos
    pos_weight = torch.tensor([n_neg / n_pos], device=device)
    print(f"\n  BCEWithLogitsLoss pos_weight = {pos_weight.item():.2f}")

    # DataLoaders
    print("\n── Sampler ──")
    sampler = make_balanced_sampler(train_ds)
    train_loader = DataLoader(
        train_ds, batch_size=cfg["batch_size"],
        sampler=sampler, num_workers=cfg["num_workers"], pin_memory=True,
    )
    test_loader = DataLoader(
        test_ds, batch_size=cfg["batch_size"],
        shuffle=False, num_workers=cfg["num_workers"], pin_memory=True,
    )

    model     = build_model(device)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = make_optimizer(model, cfg)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=cfg["epochs"], eta_min=1e-6
    )
    scaler = torch.cuda.amp.GradScaler(enabled=cfg["use_amp"])

    best_auc  = 0.0
    no_improve = 0
    history = {"loss": [], "auc": []}

    print(f"\n── Training  ({cfg['epochs']} epochs, early stop patience={cfg['patience']}) ──\n")
    t_start = time.time()

    for epoch in range(1, cfg["epochs"] + 1):
        t0 = time.time()

        # ── Train ──
        model.train()
        running_loss = 0.0
        n_batches    = 0
        it = tqdm(train_loader, desc=f"Ep {epoch:02d} train", leave=False) \
             if HAS_TQDM else train_loader

        for imgs, lbls in it:
            imgs = imgs.to(device, non_blocking=True)
            lbls = lbls.to(device, non_blocking=True).unsqueeze(1)
            optimizer.zero_grad()
            with torch.cuda.amp.autocast(enabled=cfg["use_amp"]):
                loss = criterion(model(imgs), lbls)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            running_loss += loss.item()
            n_batches    += 1

        scheduler.step()
        avg_loss = running_loss / n_batches

        # ── Evaluate ──
        model.eval()
        probs_all, labels_all = [], []
        with torch.no_grad():
            for imgs, lbls in test_loader:
                imgs = imgs.to(device, non_blocking=True)
                with torch.cuda.amp.autocast(enabled=cfg["use_amp"]):
                    p = torch.sigmoid(model(imgs).squeeze(1)).cpu().numpy()
                probs_all.extend(p)
                labels_all.extend(lbls.numpy())

        fpr, tpr, _ = roc_curve(labels_all, probs_all)
        epoch_auc   = sk_auc(fpr, tpr)
        elapsed     = time.time() - t0

        history["loss"].append(avg_loss)
        history["auc"].append(epoch_auc)

        marker = ""
        if epoch_auc > best_auc:
            best_auc   = epoch_auc
            no_improve = 0
            torch.save(model.state_dict(), cfg["checkpoint"])
            marker = "  ← best"
        else:
            no_improve += 1

        print(f"Epoch {epoch:3d}/{cfg['epochs']}  "
              f"loss={avg_loss:.4f}  AUC={epoch_auc:.4f}  "
              f"({elapsed:.1f}s){marker}")

        if no_improve >= cfg["patience"]:
            print(f"\nEarly stopping at epoch {epoch} (no AUC improvement for {cfg['patience']} epochs)")
            break

    total = time.time() - t_start
    print(f"\nDone.  Best AUC={best_auc:.4f}  Total time={total/60:.1f} min")
    return best_auc, history

In [ ]:
# 7.  Final evaluation + ROC plot
def evaluate(cfg):
    device = cfg["device"]

    test_ds = LensDataset(cfg["test_lenses"], cfg["test_nonlenses"], augment=False)
    test_loader = DataLoader(
        test_ds, batch_size=cfg["batch_size"],
        shuffle=False, num_workers=cfg["num_workers"],
    )

    model = build_model(device)
    model.load_state_dict(torch.load(cfg["checkpoint"], map_location=device))
    model.eval()

    probs_all, labels_all = [], []
    with torch.no_grad():
        for imgs, lbls in test_loader:
            imgs = imgs.to(device, non_blocking=True)
            p = torch.sigmoid(model(imgs).squeeze(1)).cpu().numpy()
            probs_all.extend(p)
            labels_all.extend(lbls.numpy())

    probs_all  = np.array(probs_all)
    labels_all = np.array(labels_all)

    fpr, tpr, thr = roc_curve(labels_all, probs_all)
    roc_auc = sk_auc(fpr, tpr)

    # Youden's J: best operating threshold
    j_scores = tpr - fpr
    best_idx = np.argmax(j_scores)
    best_thr = thr[best_idx]
    print(f"\n=== Test AUC : {roc_auc:.4f} ===")
    print(f"    Best threshold (Youden J): {best_thr:.3f}  "
          f"TPR={tpr[best_idx]:.3f}  FPR={fpr[best_idx]:.3f}")

    # Plot
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    # ROC
    axes[0].plot(fpr, tpr, color="steelblue", lw=2,
                 label=f"ResNet18  (AUC = {roc_auc:.4f})")
    axes[0].scatter(fpr[best_idx], tpr[best_idx], color="crimson", zorder=5,
                    label=f"Best threshold = {best_thr:.3f}")
    axes[0].plot([0,1],[0,1], "gray", lw=1, ls="--", label="Random")
    axes[0].set_xlabel("False Positive Rate")
    axes[0].set_ylabel("True Positive Rate")
    axes[0].set_title("ROC Curve — Gravitational Lens Detector")
    axes[0].legend(loc="lower right")
    axes[0].grid(alpha=0.3)

    # Score histogram
    axes[1].hist(probs_all[labels_all==0], bins=50, alpha=0.6,
                 color="steelblue", label="Non-lenses", density=True)
    axes[1].hist(probs_all[labels_all==1], bins=50, alpha=0.6,
                 color="crimson",   label="Lenses",     density=True)
    axes[1].axvline(best_thr, color="black", ls="--", lw=1.5,
                    label=f"Threshold = {best_thr:.3f}")
    axes[1].set_xlabel("Predicted probability")
    axes[1].set_ylabel("Density")
    axes[1].set_title("Score Distribution")
    axes[1].legend()
    axes[1].grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig(cfg["roc_plot"], dpi=150)
    print(f"Plot saved → {cfg['roc_plot']}")

    return roc_auc, fpr, tpr

In [20]:
# 8.  Entry point
if __name__ == "__main__":
    print("=" * 55)
    print("  Gravitational Lens Classifier  v2")
    print("=" * 55)

    best_auc, history = train(CFG)

    print("\n" + "=" * 55)
    print("  Final evaluation on test set")
    print("=" * 55)
    evaluate(CFG)

  Gravitational Lens Classifier  v2

── Datasets ──
  lenses=1730  non-lenses=28675  ratio=16.6:1  augment=True
  lenses=195  non-lenses=19455  ratio=99.8:1  augment=False

  BCEWithLogitsLoss pos_weight = 16.58

── Sampler ──
  Epoch size: 3460 samples  (1730 pos + 1730 neg drawn from 28675 neg)


C:\Users\dhrut\AppData\Local\Temp\ipykernel_41728\2755576419.py:34: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=cfg["use_amp"])



── Training  (40 epochs, early stop patience=7) ──



Ep 01 train:   0%|          | 0/109 [00:00<?, ?it/s]C:\Users\dhrut\AppData\Local\Temp\ipykernel_41728\2755576419.py:57: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=cfg["use_amp"]):
C:\Users\dhrut\AppData\Local\Temp\ipykernel_41728\2755576419.py:74: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=cfg["use_amp"]):


Epoch   1/40  loss=1.3676  AUC=0.9290  (73.1s)  ← best


...

Epoch  39/40  loss=0.3067  AUC=0.9813  (52.9s)


Ep 40 train:   0%|          | 0/109 [00:00<?, ?it/s]C:\Users\dhrut\AppData\Local\Temp\ipykernel_41728\2755576419.py:57: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=cfg["use_amp"]):
C:\Users\dhrut\AppData\Local\Temp\ipykernel_41728\2755576419.py:74: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=cfg["use_amp"]):


Epoch  40/40  loss=0.5112  AUC=0.9820  (54.2s)  ← best

Done.  Best AUC=0.9820  Total time=36.4 min

  Final evaluation on test set
  lenses=195  non-lenses=19455  ratio=99.8:1  augment=False


C:\Users\dhrut\AppData\Local\Temp\ipykernel_41728\3710658581.py:12: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(cfg["checkpoint"], map_loc


=== Test AUC : 0.9820 ===
    Best threshold (Youden J): 0.671  TPR=0.969  FPR=0.088
Plot saved → roc_curve_v2.png
